# 01 — Query-Generierung

Für alle **60 Songs** (50 In-DB aus `dry_ref` + 10 OOD aus `dry_ood`)
werden Distortions aus vier Gruppen erzeugt:

| Gruppe | Bedingungen | Beschreibung |
|---|---|---|
| **A** | 15 | Tempo / Pitch / Speed |
| **B** |  8 | Noise (SNR) / Room-IR / MP3 |
| **C** |  2 | Kombinierte Szenarien (Club, Radio) |
| **D** |  8 | Variable Längen × clean/SNR10 |

Jede WAV-Datei wird unter `data/queries/dry_run/` gespeichert.
Das Manifest `data/queries/manifest_dry.csv` enthält alle Pflicht-Spalten
`duration_sec` ist die **tatsächliche** Länge nach Verzerrung.

**Abhängigkeiten:** NB 00 (Partitionen in `data/partitions/`)


In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1. Imports und Pfade

`query_generation.py` enthält `generate_track_queries()` und `build_manifest()`.
Alle Distortions-Primitiven stammen aus `distortions.py`.


In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from tqdm import tqdm
from query_generation import generate_track_queries, build_manifest
from utils import load_partitions, load_fma_metadata
from run_config import cfg

RUN_MODE = cfg.run_mode
SR = 8000
SEED = 42

# ── Mode-dependent paths and partition keys ───────────────────────────────────
if RUN_MODE == "dry":
    QUERY_KEY     = cfg.query_key       # "dry_ref"
    OOD_KEY       = cfg.ood_key         # "dry_ood"
    OUT_DIR       = PROJECT_ROOT / "data" / "queries" / cfg.query_subdir
    MANIFEST_PATH = PROJECT_ROOT / "data" / "queries" / cfg.manifest_name
else:
    QUERY_KEY     = cfg.query_key       # "live_query"
    OOD_KEY       = cfg.ood_key         # "live_ood"
    OUT_DIR       = PROJECT_ROOT / "data" / "queries" / cfg.query_subdir
    MANIFEST_PATH = PROJECT_ROOT / "data" / "queries" / cfg.manifest_name

print(f"RUN_MODE      = {RUN_MODE!r}")
print(f"QUERY_KEY     = {QUERY_KEY!r}")
print(f"OOD_KEY       = {OOD_KEY!r}")
print(f"OUT_DIR       = {OUT_DIR}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")

## 2. Partitionen und Metadaten laden

Alle Track-IDs und die MUSAN-Eval-Liste stammen aus den JSONs, die NB 00
in `data/partitions/` gespeichert hat.


In [ ]:
partitions = load_partitions(PROJECT_ROOT / "data" / "partitions")
query_track_ids = partitions[QUERY_KEY]   # In-DB query songs
ood_track_ids   = partitions[OOD_KEY]     # OOD songs

with open(PROJECT_ROOT / "data" / "partitions" / "musan_split.json") as fh:
    musan_split = json.load(fh)
musan_eval_files = musan_split["eval"]

track_df = load_fma_metadata(PROJECT_ROOT / "data" / "fma_medium")
print(f"query tracks : {len(query_track_ids)}  ({'dry_ref' if RUN_MODE=='dry' else 'live_query'})")
print(f"OOD tracks   : {len(ood_track_ids)}")
print(f"MUSAN eval   : {len(musan_eval_files)} files")

if RUN_MODE == "live":
    import shutil
    n_songs = len(query_track_ids) + len(ood_track_ids)
    est_gb  = n_songs * 33 * 160 / 1024**2
    _, _, free = shutil.disk_usage(str(OUT_DIR.parent.parent))
    free_gb = free / 1024**3
    print(f"\nDisk estimate: {n_songs} songs × 33 conditions × ~160 KB ≈ {est_gb:.1f} GB")
    print(f"Free disk     : {free_gb:.1f} GB  {'✓' if free_gb > est_gb * 1.5 else 'WARNING: may be tight'}")

## 3. Impulsantwort-Dateien (Room IR)

Für den Dry Run sind keine echten Raumimpulsantworten heruntergeladen.
Als Fallback werden MUSAN-Eval-Dateien als Pseudo-IR verwendet —
sie testen den Code-Pfad vollständig, klingen aber nicht wie echte Räume.
Für den Live Run sollten hier echte IR-Dateien (z. B. AIR-Datenbank) stehen.


In [ ]:
# Dry Run:  MUSAN-Eval-Dateien als Pseudo-IRs verwenden
# Live Run: selbe Pseudo-IR-Logik (keine echten IR-Dateien im Datensatz)
ir_files = musan_eval_files

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"IR files      : {len(ir_files)}  (pseudo-IR via MUSAN eval)")
print(f"Output dir    : {OUT_DIR}")

## 4. Voraussetzungen prüfen

Stellt sicher, dass alle Audio-Dateien der Dry-Run-Tracks auf der Festplatte
vorhanden sind, bevor die zeitintensive Query-Generierung startet.


In [ ]:
all_track_ids = query_track_ids + ood_track_ids
missing = [tid for tid in all_track_ids if tid not in track_df.index]
assert len(missing) == 0, f"Tracks nicht in Metadaten: {missing}"

missing_files = [tid for tid in all_track_ids
                 if not Path(track_df.at[tid, "filepath"]).exists()]
assert len(missing_files) == 0, f"Fehlende Audio-Dateien: {missing_files}"
print(f"Alle {len(all_track_ids)} Audio-Dateien vorhanden. ✓")

## 5. In-DB Queries generieren (dry_ref)

Für jeden der 50 In-DB Tracks werden 33 WAV-Dateien erzeugt.
`ref_track_id` = `track_id` (Song ist in der Referenz-DB).


In [ ]:
indb_rows = []
skipped_indb = []
for track_id in tqdm(query_track_ids, desc="In-DB"):
    fp = track_df.at[track_id, "filepath"]
    rows = generate_track_queries(
        track_id, False, fp, str(OUT_DIR), musan_eval_files, ir_files, SR
    )
    if rows:
        indb_rows.extend(rows)
    else:
        skipped_indb.append(track_id)
print(f"In-DB rows: {len(indb_rows)}  skipped: {len(skipped_indb)}")

## 6. OOD Queries generieren (dry_ood)

Für jeden der 10 OOD Tracks werden dieselben 33 Bedingungen angewendet.
`ref_track_id` = `None` — der Song ist **nicht** in der Referenz-DB.


In [ ]:
ood_rows = []
skipped_ood = []
for track_id in tqdm(ood_track_ids, desc="OOD"):
    fp = track_df.at[track_id, "filepath"]
    rows = generate_track_queries(
        track_id, True, fp, str(OUT_DIR), musan_eval_files, ir_files, SR
    )
    if rows:
        ood_rows.extend(rows)
    else:
        skipped_ood.append(track_id)
print(f"OOD rows: {len(ood_rows)}  skipped: {len(skipped_ood)}")

## 7. Manifest zusammenstellen und speichern

`build_manifest()` erzwingt die Spaltenreihenfolge und Typen aus.
`ref_track_id` wird als nullable Int64 gespeichert, damit `None`-Werte für
OOD-Einträge korrekt als leere Felder in der CSV landen.


In [ ]:
all_rows = indb_rows + ood_rows
manifest = build_manifest(all_rows)

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Manifest gespeichert: {MANIFEST_PATH}")
print(f"Shape: {manifest.shape}")
print(manifest.head(3))


## 8. Sanity Check — Anzahl der Queries

Erwartet: 60 Songs × 33 Bedingungen = 1 980 Zeilen.
(A:15 + B:8 + C:2 + D:8 = 33 Bedingungen pro Song)


In [ ]:
n_total = len(manifest)
n_indb  = (~manifest["is_ood"]).sum()
n_ood   = manifest["is_ood"].sum()
print(f"Gesamt: {n_total}  In-DB: {n_indb}  OOD: {n_ood}")

expected_per_song = 15 + 8 + 2 + 8  # A + B + C + D = 33
n_songs_ok = (len(query_track_ids) - len(skipped_indb) + len(ood_track_ids) - len(skipped_ood))
expected_total = n_songs_ok * expected_per_song
assert n_total == expected_total, f"Erwartet {expected_total}, erhalten {n_total}"
print(f"Anzahl-Check passed: {n_total} = {n_songs_ok} Songs × {expected_per_song} ✓")

## 9. Sanity Check — OOD ref_track_id = None

OOD-Einträge müssen `ref_track_id = None` haben.
In-DB-Einträge müssen `ref_track_id = track_id` haben.


In [ ]:
ood_entries   = manifest[manifest["is_ood"]]
indb_entries  = manifest[~manifest["is_ood"]]

assert ood_entries["ref_track_id"].isna().all(),     "OOD entries must have ref_track_id = None!"
assert (indb_entries["ref_track_id"] == indb_entries["track_id"]).all(),     "In-DB entries must have ref_track_id == track_id!"
print("OOD ref_track_id = None ✓")
print("In-DB ref_track_id == track_id ✓")


## 10. Längen-Statistik Speed-Bedingungen

Speed-Bedingungen ändern die tatsächliche Länge des Audio-Signals.
`duration_sec` muss von 10.0 abweichen — das ist ein expliziter Fix (#4)
aus dem Testkonzept (kein stilles Normieren auf 10 s).


In [ ]:
speed_conds = manifest[manifest["condition"].str.startswith("A_speed_")]
stats = speed_conds.groupby("condition")["duration_sec"].agg(["mean", "min", "max"])
print("Speed-Bedingungen — tatsächliche Längen (Sekunden):")
print(stats.to_string())

# Verifikation: keine Speed-Bedingung darf exakt 10.0s haben
assert not (speed_conds["duration_sec"] == 10.0).any(),     "Speed conditions must NOT all be exactly 10.0s!"
print("\nSpeed duration != 10.0s ✓")


## 11. Überblick Gruppen und Bedingungen


In [ ]:
print("Queries pro Gruppe:")
print(manifest.groupby("group").size().to_string())
print()
print("Queries pro Condition (erste 10):")
print(manifest.groupby("condition").size().head(10).to_string())
print()
print(f"Eindeutige Bedingungen: {manifest['condition'].nunique()}")
print(f"WAV-Dateien in {OUT_DIR}:", len(list(OUT_DIR.glob('*.wav'))))
